# 🧪 Pydantic vs No-Pydantic for LLM Outputs

This notebook demonstrates **why Pydantic BaseModel matters** when working with LLM outputs.

We use the same task — asking an LLM to evaluate a job application response — and show:
- ❌ **Without Pydantic**: raw strings, brittle parsing, silent failures
- ✅ **With Pydantic**: typed, validated, safe structured output

In [1]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel, ValidationError

load_dotenv(override=True)

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY")
)
MODEL = "openai/gpt-4o-mini"

---
## 🎯 The Task

We ask the LLM to evaluate a candidate's interview answer.

We expect:
- `is_acceptable` → boolean (pass/fail)
- `score` → integer 1–10
- `feedback` → string explanation

In [2]:
SYSTEM_PROMPT = """
You are an expert hiring manager evaluating interview responses.
Given a candidate's answer, respond in JSON with:
  - is_acceptable: true or false
  - score: integer from 1 to 10
  - feedback: short explanation
"""

CANDIDATE_ANSWER = """
I believe my biggest strength is my ability to learn quickly.
In my last role, I picked up three new frameworks in a month to ship a critical feature on time.
"""

---
## ❌ WITHOUT Pydantic — Raw String Output

In [3]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": CANDIDATE_ANSWER}
    ]
)

raw_output = response.choices[0].message.content
print("=== RAW LLM OUTPUT ===")
print(raw_output)
print(f"\nType: {type(raw_output)}")

=== RAW LLM OUTPUT ===
```json
{
  "is_acceptable": true,
  "score": 8,
  "feedback": "The candidate's ability to learn quickly is a valuable strength, and providing a concrete example demonstrates practical application of this skill. However, more detail on the impact of the frameworks on the project could enhance the response."
}
```

Type: <class 'str'>


In [4]:
# 🔥 Problem 1: You have to manually parse and hope the format is consistent
try:
    # Strip markdown code fences if the LLM wraps output in ```json ... ```
    cleaned = raw_output.strip().removeprefix("```json").removesuffix("```").strip()
    parsed = json.loads(cleaned)
    print("Parsed dict:", parsed)
except json.JSONDecodeError as e:
    print(f"❌ JSON parse failed: {e}")

Parsed dict: {'is_acceptable': True, 'score': 8, 'feedback': "The candidate's ability to learn quickly is a valuable strength, and providing a concrete example demonstrates practical application of this skill. However, more detail on the impact of the frameworks on the project could enhance the response."}


In [5]:
# 🔥 Problem 2: No type safety — wrong types slip through silently
# Simulate what can happen if the LLM returns unexpected types
bad_llm_output = '{"is_acceptable": "yes", "score": "eight", "feedback": "Good answer."}'
parsed_bad = json.loads(bad_llm_output)

print("is_acceptable type:", type(parsed_bad["is_acceptable"]))  # str, not bool!
print("score type:", type(parsed_bad["score"]))                   # str, not int!

# This condition silently evaluates wrong:
if parsed_bad["is_acceptable"]:  # "yes" is truthy — but so is "no"!
    print("✅ Candidate passed")   # This prints even if LLM said 'no'!

# Arithmetic silently breaks:
try:
    average = parsed_bad["score"] / 2
except TypeError as e:
    print(f"❌ Math broke silently: {e}")

is_acceptable type: <class 'str'>
score type: <class 'str'>
✅ Candidate passed
❌ Math broke silently: unsupported operand type(s) for /: 'str' and 'int'


In [6]:
# 🔥 Problem 3: Missing fields cause KeyError crashes at runtime
incomplete_llm_output = '{"is_acceptable": true, "feedback": "Good answer."}'
parsed_incomplete = json.loads(incomplete_llm_output)

try:
    score = parsed_incomplete["score"]  # LLM forgot this field!
except KeyError as e:
    print(f"❌ Missing field crash: {e} — caught late, far from the LLM call!")

❌ Missing field crash: 'score' — caught late, far from the LLM call!


---
## ✅ WITH Pydantic — Structured, Validated, Type-Safe

In [7]:
from pydantic import BaseModel, Field, ValidationError

class Evaluation(BaseModel):
    is_acceptable: bool
    score: int = Field(ge=1, le=10)   # Must be 1-10, enforced!
    feedback: str

print("Schema:", Evaluation.model_json_schema())

Schema: {'properties': {'is_acceptable': {'title': 'Is Acceptable', 'type': 'boolean'}, 'score': {'maximum': 10, 'minimum': 1, 'title': 'Score', 'type': 'integer'}, 'feedback': {'title': 'Feedback', 'type': 'string'}}, 'required': ['is_acceptable', 'score', 'feedback'], 'title': 'Evaluation', 'type': 'object'}


In [9]:
# ✅ Use OpenAI's structured output mode — schema is passed to the LLM directly
response_structured = client.beta.chat.completions.parse(
    model=MODEL,
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": CANDIDATE_ANSWER}
    ],
    response_format=Evaluation   # Pydantic model passed directly!
)

result: Evaluation = response_structured.choices[0].message.parsed
print("=== STRUCTURED OUTPUT ===")
print(f"is_acceptable type : {type(result.is_acceptable)} → {result.is_acceptable}")
print(f"score type         : {type(result.score)} → {result.score}")
print(f"feedback type      : {type(result.feedback)} → {result.feedback}")

=== STRUCTURED OUTPUT ===
is_acceptable type : <class 'bool'> → True
score type         : <class 'int'> → 8
feedback type      : <class 'str'> → The candidate demonstrates a strong ability to learn quickly, which is valuable in a fast-paced environment. The specific example provided shows initiative and relevance to the role.


In [10]:
print(result)

is_acceptable=True score=8 feedback='The candidate demonstrates a strong ability to learn quickly, which is valuable in a fast-paced environment. The specific example provided shows initiative and relevance to the role.'


In [11]:
# ✅ Type-correct logic — no silent errors
if result.is_acceptable:  # Guaranteed bool
    print("✅ Candidate passed!")
else:
    print("❌ Candidate did not pass.")

# ✅ Arithmetic works because score is guaranteed int
normalized = result.score / 10
print(f"Normalized score: {normalized:.1%}")

✅ Candidate passed!
Normalized score: 80.0%


In [12]:
# ✅ Pydantic catches bad data IMMEDIATELY — at the boundary
print("--- Simulating bad LLM output caught by Pydantic ---")

bad_data = {"is_acceptable": "yes", "score": "eight", "feedback": "Good."}

try:
    validated = Evaluation.model_validate(bad_data)
except ValidationError as e:
    print(f"✅ Caught at the boundary:\n{e}")

--- Simulating bad LLM output caught by Pydantic ---
✅ Caught at the boundary:
1 validation error for Evaluation
score
  Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='eight', input_type=str]
    For further information visit https://errors.pydantic.dev/2.11/v/int_parsing


In [13]:
# ✅ Field constraints enforced — score out of range rejected
print("--- Score out of range ---")

out_of_range = {"is_acceptable": True, "score": 15, "feedback": "Too high!"}

try:
    validated = Evaluation.model_validate(out_of_range)
except ValidationError as e:
    print(f"✅ Caught score=15 violation:\n{e}")

--- Score out of range ---
✅ Caught score=15 violation:
1 validation error for Evaluation
score
  Input should be less than or equal to 10 [type=less_than_equal, input_value=15, input_type=int]
    For further information visit https://errors.pydantic.dev/2.11/v/less_than_equal


---
## 📊 Summary

| Problem | ❌ Without Pydantic | ✅ With Pydantic |
|---|---|---|
| **Output format** | Raw string, manual parsing | Schema enforced by LLM |
| **Wrong types** (`"yes"` instead of `true`) | Silent bug, logic errors | `ValidationError` immediately |
| **Missing fields** | `KeyError` crash at runtime | Caught at boundary |
| **Invalid ranges** (`score=15`) | No check | Field constraint rejects it |
| **Downstream code** | Fragile, defensive checks everywhere | Clean, typed `.attribute` access |
| **IDE support** | No autocomplete | Full autocomplete + type hints |

> **Rule of thumb**: Treat LLM output like user input — never trust it. Pydantic is your validation layer.